# Description

In this notebook, we benchmark different symbolic regression algorithms on the Korns benchmarks.

In [1]:
from __future__ import annotations

from pysr import PySRRegressor
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"You are using the `\^` operator, but have not set up `constraints` for it\.",
    category=UserWarning,
    module=r"pysr\.sr",
)
warnings.filterwarnings(
    "ignore",
    message=r"Note: it looks like you are running in Jupyter\. The progress bar will be turned off\.",
    category=UserWarning,
    module=r"pysr\.sr",
)

from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Protocol, Tuple, Set

import numpy as np
import sympy as sp
import h5py


N_RUNS = 1
UNARY_OPS = ["sin", "cos", "tan", "tanh", "exp", "log", "sqrt"]
BINARY_OPS = ["+", "-", "*", "/", "^"]
FEATURE_NAMES = ["x0", "x1", "x2", "x3", "x4"]


@dataclass(frozen=True)
class DatasetRecord:
    pid: str
    X: np.ndarray
    y: np.ndarray
    expr_gt: sp.Expr


def load_korns_hdf5(path: str) -> Dict[str, DatasetRecord]:
    out: Dict[str, DatasetRecord] = {}
    with h5py.File(path, "r") as f:
        for pid in f.keys():
            grp = f[pid]
            X = np.asarray(grp["X"][:], dtype=np.float64)
            y = np.asarray(grp["y"][:], dtype=np.float64).reshape(-1)
            if "expr_srepr" in grp.attrs:
                expr_gt = sp.sympify(grp.attrs["expr_srepr"])
            else:
                expr_gt = sp.sympify(grp.attrs["expr_str"])
            out[pid] = DatasetRecord(pid=pid, X=X, y=y, expr_gt=expr_gt)
    return out


def train_test_split(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.2,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = np.arange(n)
    rng.shuffle(idx)
    n_test = int(round(test_size * n))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


@dataclass(frozen=True)
class Metrics:
    nlse: float
    term_precision: float
    term_recall: float
    term_f1: float


def nlse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    err = y_true - y_pred
    mse = float(np.mean(err**2))
    var = float(np.var(y_true))
    return float(mse / var) if var > 0 else float("nan")


def _as_add_terms(expr: sp.Expr) -> Tuple[sp.Expr, ...]:
    expr = sp.sympify(expr)
    return sp.Add.make_args(expr) if expr.is_Add else (expr,)


def _strip_numeric_coefficient(term: sp.Expr) -> sp.Expr:
    term = sp.simplify(term)
    if term.is_Number:
        return sp.Integer(1)
    coeff, rest = term.as_coeff_Mul(rational=False)
    return sp.Integer(1) if rest == 1 else sp.simplify(rest)


def _canonicalize_term(term: sp.Expr) -> sp.Expr:
    t = sp.expand_mul(term)
    t = sp.expand(t)
    t = _strip_numeric_coefficient(t)
    t = sp.together(t)
    t = sp.simplify(t)
    return t


def _round_floats(expr: sp.Expr, decimals: int = 12) -> sp.Expr:
    repl = {}
    for f in expr.atoms(sp.Float):
        repl[f] = sp.Float(round(float(f), decimals))
    return expr.xreplace(repl)


def _canonicalize_symbols(expr: sp.Expr, feature_names: List[str]) -> sp.Expr:
    name_set = set(feature_names)
    repl = {}
    for s in expr.free_symbols:
        if s.name in name_set:
            repl[s] = sp.Symbol(s.name)
    return expr.xreplace(repl)


def _normalize_for_terms(expr: sp.Expr, feature_names: List[str], float_decimals: int = 12) -> sp.Expr:
    e = sp.sympify(expr)
    e = _canonicalize_symbols(e, feature_names)
    e = sp.expand_mul(e)
    e = sp.expand(e)
    e = sp.together(e)
    e = sp.simplify(e)
    e = _round_floats(e, decimals=float_decimals)
    e = sp.expand_mul(e)
    e = sp.expand(e)
    e = sp.simplify(e)
    return e


def extract_term_set(expr: sp.Expr, feature_names: List[str], float_decimals: int = 12) -> Set[sp.Expr]:
    e = _normalize_for_terms(expr, feature_names=feature_names, float_decimals=float_decimals)
    terms = _as_add_terms(e)
    const_sum = sp.Integer(0)
    nonconst_terms: List[sp.Expr] = []
    feat_name_set = set(feature_names)
    for t in terms:
        sym_names = {s.name for s in t.free_symbols}
        if sym_names.isdisjoint(feat_name_set):
            const_sum += t
        else:
            nonconst_terms.append(t)
    out: Set[sp.Expr] = set()
    const_sum = sp.simplify(const_sum)
    if const_sum != 0:
        out.add(sp.Integer(1))
    for t in nonconst_terms:
        ct = _canonicalize_term(t)
        for subt in _as_add_terms(ct):
            out.add(_canonicalize_term(subt))
    return out


def term_precision_recall_f1(expr_gt: sp.Expr, expr_pred: Optional[sp.Expr]) -> Tuple[float, float, float]:
    if expr_pred is None:
        return 0.0, 0.0, 0.0
    gt_terms = extract_term_set(expr_gt, FEATURE_NAMES, 12)
    pr_terms = extract_term_set(expr_pred, FEATURE_NAMES, 12)
    if len(pr_terms) == 0 and len(gt_terms) == 0:
        return 1.0, 1.0, 1.0
    if len(pr_terms) == 0:
        return 0.0, 0.0, 0.0
    inter = gt_terms.intersection(pr_terms)
    precision = len(inter) / len(pr_terms)
    recall = len(inter) / len(gt_terms)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return float(precision), float(recall), float(f1)


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    expr_gt: sp.Expr,
    expr_pred: Optional[sp.Expr],
) -> Metrics:
    p, r, f1 = term_precision_recall_f1(expr_gt, expr_pred)
    return Metrics(nlse=nlse(y_true, y_pred), term_precision=p, term_recall=r, term_f1=f1)


@dataclass
class SRFitResult:
    expr: Optional[sp.Expr]
    y_pred_test: np.ndarray
    metadata: Optional[Dict[str, Any]]


class SymbolicRegressor(Protocol):
    name: str
    def fit_predict(self, X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray) -> SRFitResult:
        ...


@dataclass
class DummyMeanRegressor:
    name: str = "dummy_mean"

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        mu = float(np.mean(y_train))
        y_pred = np.full(X_test.shape[0], mu, dtype=np.float64)
        return SRFitResult(expr=sp.Float(mu), y_pred_test=y_pred, metadata={"mean": mu})


@dataclass
class PySRKornsRegressor:
    name: str = "pysr"
    niterations: int = 200
    populations: int = 20
    maxsize: int = 20
    timeout_in_seconds: Optional[int] = None

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        model = PySRRegressor(
            niterations=self.niterations,
            populations=self.populations,
            maxsize=self.maxsize,
            unary_operators=UNARY_OPS,
            binary_operators=BINARY_OPS,
            elementwise_loss="loss(x, y) = (x - y)^2",
            model_selection="best",
            verbosity=0,
            progress=False,
            temp_equation_file=True,
            delete_tempfiles=True,
            timeout_in_seconds=self.timeout_in_seconds,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        try:
            expr = model.sympy()
        except Exception:
            expr = None
        return SRFitResult(
            expr=expr,
            y_pred_test=np.asarray(y_pred, dtype=np.float64),
            metadata={
                "niterations": self.niterations,
                "populations": self.populations,
                "maxsize": self.maxsize,
            },
        )


@dataclass
class RunConfig:
    hdf5_path: str = "korns_dataset.hdf5"
    test_size: float = 0.2
    split_seed: int = 0
    per_problem_seed_offset: int = 1000
    algo_seed_offset: int = 10_000
    run_seed_offset: int = 1_000_000


@dataclass
class BenchmarkRow:
    pid: str
    algo: str
    run_id: int
    runtime_sec: float
    nlse: float
    term_precision: float
    term_recall: float
    term_f1: float
    expr_str: str
    expr_gt_str: str
    extra: Optional[Dict[str, Any]] = None


def zlib_crc32(b: bytes) -> int:
    import zlib
    return zlib.crc32(b) & 0xFFFFFFFF


def _get_algo_seed(algo_name: str) -> int:
    return int(np.uint32(zlib_crc32(algo_name.encode("utf-8"))))


def run_benchmark(datasets, algorithms, config, n_runs=N_RUNS):
    rows = []
    for pid, rec in sorted(datasets.items(), key=lambda kv: int(kv[0][1:])):
        split_seed = config.split_seed + config.per_problem_seed_offset + int(pid[1:])
        X_train, X_test, y_train, y_test = train_test_split(rec.X, rec.y, config.test_size, split_seed)
        print("=" * 50)
        print(f"[PROBLEM] {pid}")
        print(f"[GT] {rec.expr_gt}")
        for algo in algorithms:
            algo_seed = _get_algo_seed(algo.name)
            print("=" * 50)
            print(f"[ALGO] {algo.name}")
            for run_id in range(n_runs):
                run_seed = (
                    config.split_seed
                    + config.per_problem_seed_offset * int(pid[1:])
                    + config.algo_seed_offset * algo_seed
                    + config.run_seed_offset * run_id
                )
                print(f"[RUN START] run_id={run_id} seed={run_seed}")
                fit_res = algo.fit_predict(X_train, y_train, X_test)
                m = compute_metrics(y_test, fit_res.y_pred_test, rec.expr_gt, fit_res.expr)
                expr_pred = str(fit_res.expr) if fit_res.expr is not None else ""
                print(f"[PRED] {expr_pred}")
                rows.append(
                    BenchmarkRow(
                        pid=pid,
                        algo=algo.name,
                        run_id=run_id,
                        runtime_sec=0.0,
                        nlse=m.nlse,
                        term_precision=m.term_precision,
                        term_recall=m.term_recall,
                        term_f1=m.term_f1,
                        expr_str=expr_pred,
                        expr_gt_str=str(rec.expr_gt),
                        extra={"run_seed": run_seed},
                    )
                )
    return rows


def save_results_csv(rows, path):
    import csv
    fieldnames = list(asdict(rows[0]).keys()) if rows else []
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            d = asdict(r)
            if d.get("extra") is not None:
                d["extra"] = str(d["extra"])
            w.writerow(d)


if __name__ == "__main__":
    cfg = RunConfig()
    datasets = load_korns_hdf5(cfg.hdf5_path)
    algos = [
        DummyMeanRegressor(),
        PySRKornsRegressor(niterations=1, populations=30, maxsize=25),
    ]
    rows = run_benchmark(datasets, algos, cfg, N_RUNS)
    save_results_csv(rows, "korns_benchmark_results_v1.csv")


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
[PROBLEM] P1
[GT] 24.3*x3 + 1.57
[ALGO] dummy_mean
[RUN START] run_id=0 seed=38912589021000
[PRED] -1.34446068188844
[ALGO] pysr
[RUN START] run_id=0 seed=28732350661000
[PRED] x3 + (x3 + x3)*11.65 - 1*(-1.5699999)
[PROBLEM] P2
[GT] 0.23 + 4.73333333333333*(x1 + x3)/x4
[ALGO] dummy_mean
[RUN START] run_id=0 seed=38912589022000
[PRED] 0.595200925459295
[ALGO] pysr
[RUN START] run_id=0 seed=28732350662000
[PRED] tanh(x4/((x3*0.010065178)))*7.052714
[PROBLEM] P3
[GT] -5.41 + 1.63333333333333*(-x0 + x1/x4 + x3)/x4
[ALGO] dummy_mean
[RUN START] run_id=0 seed=38912589023000
[PRED] -5.23526279586559
[ALGO] pysr
[RUN START] run_id=0 seed=28732350663000
[PRED] -x0/x4 - 5.226692
[PROBLEM] P4
[GT] 0.13*sin(x2) - 2.3
[ALGO] dummy_mean
[RUN START] run_id=0 seed=38912589024000
[PRED] -2.30004270955445
[ALGO] pysr
[RUN START] run_id=0 seed=28732350664000
[PRED] cos(x0)*0.07099731/(-45.675